# Scientific Reports successor — Step 4

Reciprocal weak-anisotropy hierarchy and six-artery validity domain. The notebook consumes a passing Step 3 result, derives the perturbation coefficients, executes the publication sweep and stops before the interaction kernel.

In [ ]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'successor/scirep-waveform-susceptibility'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_root = Path('/content/picoNewton')
    if not repo_root.exists():
        subprocess.run(['git','clone','https://github.com/khalid-saqr/picoNewton.git',str(repo_root)],check=True)
    subprocess.run(['git','-C',str(repo_root),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(repo_root),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
    study_root = Path('/content/drive/MyDrive/picoNewton_susceptibility')
else:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root/'picoNewton_v3').exists():
        repo_root = repo_root.parent
    study_root = repo_root/'piconewton_susceptibility_outputs'
print({'repo_root':str(repo_root),'study_root':str(study_root),'colab':IN_COLAB})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'picoNewton_v3')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'piconewton_susceptibility')],check=True)

## Resolve prior gates

A clean Drive can reconstruct Step 2 and Step 3. Existing passing outputs are reused after their fail-closed validators inspect them.

In [ ]:
step2_root = study_root/'bootstrap'/'step2'
step3_root = study_root/'step3_parent_continuity'
step4_root = study_root/'step4_perturbation'
if not (step2_root/'completion_gate.json').exists():
    subprocess.run([
        'piconewton-susceptibility-bootstrap','--repo-root',str(repo_root),
        '--storage','local','--local-root',str(study_root)
    ],check=True)
if not (step3_root/'step3_manifest.json').exists():
    subprocess.run([
        'piconewton-susceptibility-step3','--step2-root',str(step2_root),
        '--output',str(step3_root),'--profile','publication'
    ],check=True)
print({'step2_root':str(step2_root),'step3_root':str(step3_root),'step4_root':str(step4_root)})

## Execute Step 4 publication profile

In [ ]:
subprocess.run([
    'piconewton-susceptibility-step4','--step3-root',str(step3_root),
    '--output',str(step4_root),'--profile','publication'
],check=True)

In [ ]:
import json, pandas as pd
manifest=json.loads((step4_root/'step4_manifest.json').read_text())
validity=pd.read_csv(step4_root/'validity_domains.csv')
slopes=pd.read_csv(step4_root/'order_slopes.csv')
print(json.dumps(manifest['gates'],indent=2,sort_keys=True))
display(validity)
display(slopes)

## Stop boundary

A passing manifest authorizes Step 5. This notebook does not construct the harmonic-interaction kernel or any critical-anisotropy inversion.